# Sprint 4 Experiment — Mistral Small 2603 (OpenRouter) × Gemini Failures

This notebook runs the 76 questions that Gemini 3.1 Flash-Lite failed on through
Mistral Small 2603 via OpenRouter, with reasoning enabled.

| Setting | Value |
|---------|-------|
| Model | mistralai/mistral-small-2603 (OpenRouter) |
| Questions | gemini_simple_failures.csv (76 rows — Gemini WRONG + REFUSED + NO ANSWER) |
| Prompt | simple (zero-shot, same as Gemini baseline) |
| Reasoning | enabled (OpenRouter reasoning API) |
| Chunk size | 3000 (author default — fixed) |
| Chunk overlap | 300 (author default — fixed) |
| Top-K | 5 (author default — fixed) |
| Temperature | 0.0 (professor requirement — fixed) |
| Embedding | all-MiniLM-L6-v2 (local, free) |

## Cell 1 — Setup paths

In [1]:
import sys, os

project_root = '/Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026'
sprint4_root = os.path.join(project_root, 'Sprint 4')
sprint3_uda  = os.path.join(project_root, 'Sprint 3', 'UDA-Benchmark')

for p in [sprint4_root, sprint3_uda]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(sprint3_uda)
print(f'project_root : {project_root}')
print(f'sprint4_root : {sprint4_root}')
print(f'sprint3_uda  : {sprint3_uda}')
print(f'cwd          : {os.getcwd()}')

if not os.path.isdir(sprint3_uda):
    raise FileNotFoundError(f'sprint3_uda not found: {sprint3_uda}')

project_root : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026
sprint4_root : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 4
sprint3_uda  : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark
cwd          : /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 3/UDA-Benchmark


## Cell 2 — Load Gemini failures

In [2]:
import pandas as pd

MODEL_KEY = 'mistral-small-2603'
PROMPT    = 'simple'

FAILURES_CSV = os.path.join(
    sprint4_root,
    'experiments/gemini-3.1-flash-lite/results/gemini_simple_failures.csv'
)
OUTPUT_DIR = os.path.join(sprint4_root, f'experiments/{MODEL_KEY}/results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALL_QUESTIONS_CSV = os.path.join(sprint4_root, 'benchmark/questions/all_questions_combined.csv')

UDA_PDF_BASE = os.path.join(sprint3_uda, 'dataset/src_doc_files_example')
PDF_DIRS = {
    'music_structured': os.path.join(UDA_PDF_BASE, 'music_docs'),
    'tathybrid':        os.path.join(UDA_PDF_BASE, 'tat_docs'),
    'finhybrid':        os.path.join(UDA_PDF_BASE, 'fin_docs'),
    'nqtext':           os.path.join(UDA_PDF_BASE, 'wiki_nq_docs/pdfs'),
    'fetatab':          os.path.join(UDA_PDF_BASE, 'wiki_feta_docs/pdfs'),
    'papertab':         os.path.join(UDA_PDF_BASE, 'paper_docs'),
    'papertext':        os.path.join(UDA_PDF_BASE, 'paper_docs'),
}

df_fail = pd.read_csv(FAILURES_CSV)
df_all  = pd.read_csv(ALL_QUESTIONS_CSV)

# Merge doc_name in (failures CSV doesn't have it)
df_fail = df_fail.merge(
    df_all[['question_id', 'doc_name']],
    on='question_id',
    how='left'
)

print(f'Gemini failures loaded: {len(df_fail)} questions')
print()
print('By dataset:')
print(df_fail['dataset'].value_counts().to_string())
print()
print('By failure_type:')
print(df_fail['failure_type'].value_counts().to_string())
print()

# Verify PDFs
print('PDF availability check:')
missing = []
for _, row in df_fail.iterrows():
    pdf_dir = PDF_DIRS.get(row['dataset'], '')
    pdf_path = os.path.join(pdf_dir, str(row['doc_name']) + '.pdf')
    if not os.path.exists(pdf_path):
        missing.append(f"{row['dataset']}/{row['doc_name']}")
missing = list(dict.fromkeys(missing))
if missing:
    print(f'  WARNING — {len(missing)} PDFs not found:')
    for p in missing: print(f'    {p}')
else:
    print(f'  All PDFs found.')

Gemini failures loaded: 76 questions

By dataset:
dataset
finhybrid           31
tathybrid           29
music_structured    16

By failure_type:
failure_type
REFUSED      38
WRONG        37
NO ANSWER     1

PDF availability check:
  All PDFs found.


## Cell 3 — Run Mistral on Gemini failures

In [3]:
from framework.rag_runner import RAGRunner
from datetime import datetime

all_results = []

for dataset, group_df in df_fail.groupby('dataset'):
    print(f"\n{'='*60}")
    print(f'Dataset: {dataset}  ({len(group_df)} questions)')
    print(f"{'='*60}")

    pdf_dir = PDF_DIRS.get(dataset, '')
    if not pdf_dir or not os.path.isdir(pdf_dir):
        print(f'  SKIP — PDF directory not found: {pdf_dir}')
        continue

    tmp_csv = os.path.join(OUTPUT_DIR, f'_tmp_{dataset}.csv')
    group_df.to_csv(tmp_csv, index=False)

    runner_dataset = dataset if dataset != 'music_structured' else 'nqtext'

    runner = RAGRunner(model_key=MODEL_KEY, dataset=runner_dataset, prompt=PROMPT)
    results_df = runner.run(
        questions_csv=tmp_csv,
        pdf_dir=pdf_dir,
        doc_col='doc_name',
        output_dir=OUTPUT_DIR,
    )
    results_df['dataset_actual'] = dataset
    all_results.append(results_df)
    os.remove(tmp_csv)

results_combined = pd.concat(all_results, ignore_index=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
final_path = os.path.join(OUTPUT_DIR, f'ALL_RESULTS_{MODEL_KEY}_{PROMPT}_{ts}.csv')
results_combined.to_csv(final_path, index=False)
print(f'\nAll results saved → {final_path}')
print(f'Total rows: {len(results_combined)}')

/Users/I772947/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/I772947/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and t


Dataset: finhybrid  (31 questions)


/Users/I772947/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAGRunner ready: model=mistral-small-2603, dataset=finhybrid, prompt=simple
  CHUNK_SIZE=3000, CHUNK_OVERLAP=300, TOP_K=5, TEMP=0.0

  PDF: ABMD_2012.pdf
  110 chunks indexed
  [1/7] during the 2012 year , did the equity awards in which the prescribed p...
    → 
  [2/7] for equity awards where the performance criteria has been met in 2012 ...
    → 
  [3/7] did abiomed outperform the nasdaq medical equipment index?...
    → 
  [4/7] did abiomed outperform the nasdaq composite index?...
    → 
  [5/7] how much of total future minimum lease payments are due currently?...
    → The answer is: 1,473
  [6/7] what is the roi of an investment in abiomed inc from march 2007 to mar...
    → 
  [7/7] what is the roi of an investment in nasdaq composite index from march ...
    → 

  PDF: ADI_2009.pdf
  141 chunks indexed
  [1/6] what is the the interest expense in 2009?...
    → The answer is: 4,094
  [2/6] what is the expected growth rate in amortization expense in 2010?...
    → The answer is

## Cell 4 — Compare Mistral vs Gemini (side by side)

In [4]:
comparison = results_combined[['question_id', 'dataset_actual', 'question', 'ground_truth', 'response']].copy()
comparison = comparison.rename(columns={'response': 'mistral_answer', 'dataset_actual': 'dataset'})

# Merge Gemini's answer back in
gemini_col = 'llm_answer' if 'llm_answer' in df_fail.columns else 'gemini_answer'
comparison = comparison.merge(
    df_fail[['question_id', gemini_col, 'failure_type']].rename(columns={gemini_col: 'gemini_answer'}),
    on='question_id', how='left'
)

comparison['mistral_empty'] = comparison['mistral_answer'].fillna('').str.strip() == ''
print('=== Mistral answer rate by dataset ===')
summary = comparison.groupby('dataset').agg(
    total=('question_id','count'),
    answered=('mistral_empty', lambda x: (~x).sum()),
    empty=('mistral_empty','sum'),
).reset_index()
summary['answer_rate'] = (summary['answered'] / summary['total'] * 100).round(1)
print(summary.to_string(index=False))

cmp_path = os.path.join(OUTPUT_DIR, f'comparison_gemini_vs_mistral_{ts}.csv')
comparison.to_csv(cmp_path, index=False)
print(f'\nComparison saved → {cmp_path}')

=== Mistral answer rate by dataset ===
         dataset  total  answered  empty  answer_rate
       finhybrid     31        19     12         61.3
music_structured     16         9      7         56.2
       tathybrid     29        24      5         82.8

Comparison saved → /Users/I772947/personal work/LLM Benchmark Team Project/LLM_Benchmark_Team_Project_2026/Sprint 4/experiments/mistral-small-2603/results/comparison_gemini_vs_mistral_20260720_121455.csv


## Cell 5 — Sample answers (spot check)

In [5]:
pd.set_option('display.max_colwidth', 120)
for _, row in comparison.head(10).iterrows():
    print(f"\n--- {row['question_id']} [{row['dataset']}] [{row['failure_type']}] ---")
    print(f"Q  : {row['question']}")
    print(f"GT : {row['ground_truth']}")
    print(f"GEM: {str(row['gemini_answer'])[:150]}")
    print(f"MIS: {str(row['mistral_answer'])[:150]}")


--- S3_FINHYBRID_ABMD/2012/page_75.pdf-1 [finhybrid] [WRONG] ---
Q  : during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
GT : yes
GEM: The provided text states that for the year ended March 31, 2012, the Company recorded $3.3 million in stock-based compensation expense for equity awar
MIS: 

--- S3_FINHYBRID_ABMD/2012/page_75.pdf-2 [finhybrid] [WRONG] ---
Q  : for equity awards where the performance criteria has been met in 2012 , what is the average compensation expense per year over which the cost will be expensed?
GT : 1719526 | 1714285.71429
GEM: The provided text states that the remaining unrecognized compensation expense for equity awards as of March 31, 2012, is $3.6 million, and the weighte
MIS: 

--- S3_FINHYBRID_ABMD/2012/page_41.pdf-2 [finhybrid] [REFUSED] ---
Q  : did abiomed outperform the nasdaq medical equipment index?
GT : yes | yes
GE